# Jupyter Notebook: From Raw Loops to LangChain Pipelines
This notebook demonstrates how to upgrade a manual "LLM-as-a-Judge" workflow to a structured LangChain pipeline.

## What is LangChain?
LangChain is a Python framework that makes it easy to build workflows around large language models by handling retrieval, document loading, chunking, embeddings, prompt templates, and multi-step pipelines. For computational social science, it is useful because it lets researchers efficiently process large text corpora (scraping, cleaning, embedding, retrieving), run structured LLM analyses at scale (classification, summarization, etc.), and integrate external data sources or tools.

## 1. Setup and Imports
First, we import the necessary libraries. Notice we are bringing in pydantic to define our data structure.

In [ ]:
%pip install -U langchain langchain-community langchain-core langchain-ollama langchain-google-genai

In [2]:
import pandas as pd
import random
from typing import Literal, Optional

# LangChain Core
from pydantic import BaseModel, Field

# LangChain Models (You can easily swap these!)
from langchain_ollama import ChatOllama  # local models
from langchain_google_genai import ChatGoogleGenerativeAI  # Google GenAI
from langchain_core.prompts import ChatPromptTemplate

/Users/tomvannuenen/anaconda3/envs/dlab2/lib/python3.10/site-packages/google/api_core/_python_version_support.py:266: FutureWarning: You are using a Python version (3.10.18) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


## 2. Defining Structured Output

Let's define a Schema with Pydantic. This forces the LLM to output strictly formatted JSON.

In [4]:
from pydantic import BaseModel, Field
from typing import Literal
import json

class AitaEvaluation(BaseModel):
    """
    Evaluation schema for AITA moral dilemmas.
    """
    verdict: Literal["YTA", "NTA", "ESH", "NAH", "INFO"] = Field(
        ..., 
        description=(
            "The categorical judgment of the scenario: "
            "YTA (You're the Asshole): OP is at fault. "
            "NTA (Not the Asshole): OP is NOT to blame. "
            "ESH (Everyone Sucks Here): Both parties are to blame. "
            "NAH (No Assholes Here): Neither party is to blame. "
            "INFO (Not Enough Info): Missing details."
        )
    )
    reasoning: str = Field(
        ..., 
        description="A single concise paragraph explaining why you chose this label."
    )

# Let's see what this schema looks like (this is what is sent to the LLM under the hood)
print(json.dumps(AitaEvaluation.model_json_schema(), indent=2))

{
  "description": "Evaluation schema for AITA moral dilemmas.",
  "properties": {
    "verdict": {
      "description": "The categorical judgment of the scenario: YTA (You're the Asshole): OP is at fault. NTA (Not the Asshole): OP is NOT to blame. ESH (Everyone Sucks Here): Both parties are to blame. NAH (No Assholes Here): Neither party is to blame. INFO (Not Enough Info): Missing details.",
      "enum": [
        "YTA",
        "NTA",
        "ESH",
        "NAH",
        "INFO"
      ],
      "title": "Verdict",
      "type": "string"
    },
    "reasoning": {
      "description": "A single concise paragraph explaining why you chose this label.",
      "title": "Reasoning",
      "type": "string"
    }
  },
  "required": [
    "verdict",
    "reasoning"
  ],
  "title": "AitaEvaluation",
  "type": "object"
}


## 3. LangChain as a "Universal Translator"

Here we build the pipeline. 

How is LangChain different from our previous approaches? As you may have noticed, with a manual approach, we have to rewrite the API call bsaed on the model we are calling:
- Ollama uses: `format=schema`
- OpenAI uses: `response_format={"type": "json_object"}` (or tools depending on the method)
- Anthropic uses: `tools=[{...}]`
- Google Gemini uses: `response_mime_type="application/json"`

With LangChain, we just change **one line of code**. The `with_structured_output()` method detects which model we are using and automatically translates the Pydantic class into the specific API format that vendor requires (JSON mode vs. Tool Calling).

Also note that our LangChain prompts are separated into "system" and "human" prompts. When you use ("human", "{text}"), LangChain automatically translates that into the standard `{"role": "user", "content": "..."}` format that APIs like Ollama expect. It translates between the different quirks that APIs have:

- Anthropic's API crashes if you try to send a "system" message inside the normal message list. LangChain extracts the system message from your list and moves it to the correct API field.
- Google doesn't use the role "assistant"; it uses "model". LangChain renames the roles for you.

Why the different name? LangChain uses abstract names (human, ai) to standardize different API terminologies. "human" becomes `role: "user"` (in Ollama/OpenAI). "ai" becomes `role: "assistant"` (in Ollama/OpenAI).

In [5]:
# 1. Initialize the Model
# Switch between Ollama and other vendors (e.g. OpenAI) by changing 1 line. No other code changes needed.
llm = ChatOllama(model="llama3.2", temperature=.4) 
# llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash", temperature=0.4)

# 2. Bind the Schema (The Magic Step)
# This tells the LLM: "Your output MUST match the AitaEvaluation class."
structured_llm = llm.with_structured_output(AitaEvaluation)

# 3. Create the Prompt Template
system_prompt = """
You are an expert at evaluating moral dilemmas from Reddit.
Analyze the user's story and determine who is at fault.
Be objective and fair.
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{text}")
])

# 4. Connect them into a runnable chain
chain = prompt | structured_llm

## 4. Running the Chain (Batch Processing)
Instead of writing a for loop, we can use `.batch()`. LangChain optimizes this.

In [6]:
df = pd.read_csv("../../data/aita_top_subs.csv")
sample_df = df.sample(3, random_state=42).reset_index(drop=True)
sample_texts = sample_df["selftext"].tolist()

In [7]:
print(sample_texts[0])

This sounds so petty and I swear I wouldn't have posted here if my boyfriend didn't practically blow up on me and refuse to come to any logical agreement.

I have very bad rosacea, seasonal psoriasis, as well as keratosis pilaris. All this means I use relatively expensive treatments and products to aid my skin. But for whatever reason, my boyfriend just won't stop using these, even though he doesn't have any of the skin issues I do. I've tried speaking to him about this, I've even bought him some nice skincare which has remained untouched because he wants my expensive stuff.

So I decided to move everything from our upstairs bathroom to our downstairs bathroom and that's where I'll be showering from now on.

When my boyfriend found out he totally blew up on me and called me an unreasonable asshole. I said he can go use the heaps of things I bought him, but he doesn't want it.

AITA for moving all my skincare and bath products to the other bathroom?


In [8]:
# Prepare the inputs
print(f"Processing {len(sample_texts)} posts...")

# Run inference on all posts at once
results = chain.batch([{"text": post} for post in sample_texts])

# 'results' is now a list of AitaEvaluation objects, not strings!
print(f"First result type: {type(results[0])}")

Processing 3 posts...
First result type: <class '__main__.AitaEvaluation'>


## 5. Analysis: Converting back to DataFrames
Because the output is structured objects, we can instantly convert it back to a Pandas DataFrame for analysis. No Regex or string parsing required.

In [9]:
# Extract data from the Pydantic objects
results_data = [res.model_dump() for res in results]

# Create a results DataFrame
results_df = pd.DataFrame(results_data)

# Combine with original text for viewing
final_df = pd.concat([sample_df, results_df], axis=1)

# Display
final_df

,idint,idstr,created,self,nsfw,author,title,url,selftext,score,...,flair_css_class,augmented_at,augmented_count,created_date,year,month,day_of_week,text_length,verdict,reasoning
0,1301653638,t3_liyydi,1613216032,1.0,0.0,skincareAITA,AITA for taking all my skincare and bath produ...,NaN,This sounds so petty and I swear I wouldn't ha...,6166.0,...,not,NaN,NaN,2021-02-13 11:33:52,2021,2,Saturday,962.0,NAH,While your boyfriend's reaction was unacceptab...
1,1177930910,t3_jhb5b2,1603554479,1.0,0.0,amianasshole1234,WIBTA if I uninvite my dad and step mom to my ...,NaN,Throwaway for privacy purposes. \n\nI’m engage...,7848.0,...,not,NaN,NaN,2020-10-24 15:47:59,2020,10,Saturday,2130.0,NTA,The user's fiancé is at fault for not being mo...
2,1060014771,t3_hj3smr,1593579688,1.0,0.0,Zealousideal-Ad-8883,AITA for locking my wife out of my office beca...,NaN,I have been studying for a very important test...,16043.0,...,not,NaN,NaN,2020-07-01 05:01:28,2020,7,Wednesday,1669.0,NAH,While it's understandable that you're feeling ...


## 6. Advanced Reliability: The "Jury" System (LangGraph)

In research, relying on a single LLM generation can be risky because models are probabilistic. They might give a different answer if you run them twice. Relying on their self-reported "confidence score" is also scientifically dubious, as models are often "confidently wrong."

A more rigorous approach is **Consensus Voting** (often called Self-Consistency). This mimics Inter-Coder Reliability in qualitative research.

### What is LangGraph?

LangGraph is a framework designed for building complex, non-linear workflows that require loops or conditional logic, rather than just straight-line execution. Its core architectural component is the **State**, a shared data structure (sort of a dictionary) that persists throughout the entire lifecycle of the graph. Instead of functions passing variables directly to one another, every "node" in the graph receives the current State as input, performs a specific task, and returns updates to modify that State. In our Jury system example, this allows the workflow to centrally store the original post, accumulate the three votes from the first step, and make all that data accessible to the final tie-breaker step without complex data passing.

In this workflow, we will use LangGraph to build a non-linear pipeline:

- The Jury: We ask the model to classify the post 3 separate times (with a slight temperature to allow for variation).
- The Vote: We count the results.
- Majority (2-1 or 3-0): We accept the consensus.
- Hung Jury (1-1-1): If the model gives three different answers (e.g., YTA, NTA, ESH), we route the post to a "Tie-Breaker" step (a stricter prompt or a better model).

This ensures that your final dataset only contains robust, reproducible classifications.

In [52]:
from collections import Counter
from typing import TypedDict, List, Annotated
from langgraph.graph import StateGraph, END

# 1. Define the State
# This acts as the "memory" that is passed between steps
class JuryState(TypedDict):
    text: str               # The input post
    votes: List[str]        # A list to store the 3 votes, e.g. ['YTA', 'YTA', 'NTA']
    final_verdict: str      # The final decision

In [53]:
# 2. Define the Nodes (The Workers)

def jury_node(state: JuryState):
    """
    Asks the model to vote 3 times. 
    We increase temperature slightly to ensure the model isn't just repeating itself.
    """
    
    # Temperature 0.4 allows for slight creative variation
    # We use the same AitaEvaluation schema from earlier
    voter_llm = llm.with_structured_output(AitaEvaluation)
    
    votes = []
    for i in range(3):
        result = voter_llm.invoke(state['text'])
        votes.append(result.verdict)
    
    return {"votes": votes}

def tie_breaker_node(state: JuryState):
    """
    This only runs if the jury cannot agree.
    We ask the model to synthesize the conflict.
    """
    print("--- JURY HUNG: Calling Tie-Breaker ---")
    
    prompt = f"""
    Review this reddit post. There was a disagreement on the verdict.
    Votes so far: {state['votes']}.
    Make a final, objective decision.
    Post: {state['text']}
    """
    
    # Using temperature 0 for the final authoritative decision
    final_judge = llm.with_structured_output(AitaEvaluation)
    result = final_judge.invoke(prompt)
    
    return {"final_verdict": result.verdict}

def simple_majority_node(state: JuryState):
    """
    Extracts the winner from the votes.
    """
    vote_counts = Counter(state["votes"])
    winner = vote_counts.most_common(1)[0][0]
    return {"final_verdict": winner}

In [54]:
# 3. Define the Routing Logic
def check_vote_count(state: JuryState):
    """
    Decides where to go next based on the vote split.
    """
    counts = Counter(state["votes"])
    top_vote_count = counts.most_common(1)[0][1] # How many votes did the winner get?
    
    if top_vote_count >= 2:
        return "majority" # 2-1 or 3-0
    else:
        return "split"    # 1-1-1 (Everyone disagrees)

In [55]:
# 4. Build the Graph
workflow = StateGraph(JuryState)

# Add the nodes
workflow.add_node("jury", jury_node)
workflow.add_node("finalize", simple_majority_node)
workflow.add_node("tie_breaker", tie_breaker_node)

# Define the flow
workflow.set_entry_point("jury")

# Conditional Logic: After Jury, check the votes
workflow.add_conditional_edges(
    "jury",
    check_vote_count,
    {
        "majority": "finalize",   # If agreed, go to finalize
        "split": "tie_breaker"    # If split, go to tie breaker
    }
)

# Both paths eventually end
workflow.add_edge("finalize", END)
workflow.add_edge("tie_breaker", END)

# Compile
app = workflow.compile()

In [57]:
import textwrap

print(textwrap.fill(sample_texts[0], width=100))

# Run the Jury System on 1 sample
result = app.invoke({"text": sample_texts[0]})

print(f"Votes: {result['votes']}")
print(f"Final Verdict: {result['final_verdict']}")

This sounds so petty and I swear I wouldn't have posted here if my boyfriend didn't practically blow
up on me and refuse to come to any logical agreement.  I have very bad rosacea, seasonal psoriasis,
as well as keratosis pilaris. All this means I use relatively expensive treatments and products to
aid my skin. But for whatever reason, my boyfriend just won't stop using these, even though he
doesn't have any of the skin issues I do. I've tried speaking to him about this, I've even bought
him some nice skincare which has remained untouched because he wants my expensive stuff.  So I
decided to move everything from our upstairs bathroom to our downstairs bathroom and that's where
I'll be showering from now on.  When my boyfriend found out he totally blew up on me and called me
an unreasonable asshole. I said he can go use the heaps of things I bought him, but he doesn't want
it.  AITA for moving all my skincare and bath products to the other bathroom?
Votes: ['NTA', 'YTA', 'NTA']
Final Ve

In [58]:
from tqdm import tqdm

# Run jury analysis on samples and add to dataframe
jury_verdicts = []

for text in tqdm(sample_df['selftext']):
    result = app.invoke({"text": text})
    jury_verdicts.append(result['final_verdict'])

# Add to dataframe
sample_df['jury_verdict'] = jury_verdicts

100%|██████████| 3/3 [00:42<00:00, 14.24s/it]


In [59]:
sample_df

,idint,idstr,created,self,nsfw,author,title,url,selftext,score,...,flair_text,flair_css_class,augmented_at,augmented_count,created_date,year,month,day_of_week,text_length,jury_verdict
0,1301653638,t3_liyydi,1613216032,1.0,0.0,skincareAITA,AITA for taking all my skincare and bath produ...,NaN,This sounds so petty and I swear I wouldn't ha...,6166.0,...,Not the A-hole,not,NaN,NaN,2021-02-13 11:33:52,2021,2,Saturday,962.0,YTA
1,1177930910,t3_jhb5b2,1603554479,1.0,0.0,amianasshole1234,WIBTA if I uninvite my dad and step mom to my ...,NaN,Throwaway for privacy purposes. \n\nI’m engage...,7848.0,...,Not the A-hole,not,NaN,NaN,2020-10-24 15:47:59,2020,10,Saturday,2130.0,YTA
2,1060014771,t3_hj3smr,1593579688,1.0,0.0,Zealousideal-Ad-8883,AITA for locking my wife out of my office beca...,NaN,I have been studying for a very important test...,16043.0,...,Not the A-hole,not,NaN,NaN,2020-07-01 05:01:28,2020,7,Wednesday,1669.0,NAH
